In [29]:
pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 44.6 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


# 2. Implicit example

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .appName("SparkByExamples")\
    .config("spark.executor.memory", "16g") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.instances", "10") \
    .getOrCreate()
    # .config("spark.some.config.option", "config-value") 

In [3]:
executor_memory = spark.sparkContext.getConf().get("spark.executor.memory")
print(f"Executor Memory: {executor_memory}")

Executor Memory: 16g


In [415]:
driver_memory = spark.sparkContext.getConf().get("spark.driver.memory")
print(f"driver Memory: {driver_memory}")

driver Memory: 2g


In [8]:
spark.sparkContext.getConf().get("spark.executor.instances")

'10'

In [10]:
executor_cores = spark.sparkContext.getConf().get("spark.executor.cores", "Not Set")
print(f"Executor Cores: {executor_cores}")

Executor Cores: Not Set


In [ ]:
for conf in spark.sparkContext.getConf().getAll():
    print(conf)

In [4]:
# BUCKET = "fgao-ensae"
BUCKET = "ematzner-ensae"
FILE_KEY_S3 = "Spark/Recommendation/"
s3_path = f"s3a://{BUCKET}/{FILE_KEY_S3}"

In [10]:
s3_path

's3a://fgao-ensae/Data_spark/Recommendation/'

In [5]:
artist_by_id = spark.read.option("delimiter", "\t").csv(s3_path+'artist_data_small.txt')

In [6]:
column_names = ["artist", "name"]
artist_by_id = artist_by_id.toDF(*column_names)

In [7]:
artist_by_id.count()
artist_by_id.show()

+-------+--------------------+
| artist|                name|
+-------+--------------------+
|1240105|        André Visior|
|1240113|           riow arai|
|1240132|Outkast & Rage Ag...|
|6776115|            小松正夫|
|1030848|      Raver's Nature|
|6671601|      Erguner, Kudsi|
|1106617|              Bloque|
|1240185|      Lexy & K. Paul|
|6671631|    Rev. W.M. Mosley|
|6671632|      Labelle, Patti|
|1240238|   the Chinese Stars|
|1240262|            The Gufs|
|6718605|          Bali Music|
|6828988|Southern Conferen...|
|1240415|        Paul & Paula|
|1009439|            Cinnamon|
|1018275|      School Of Fish|
|6671680|Armstrong, Louis ...|
|1240508|The Ozark Mountai...|
|1240510| The Mercury Program|
+-------+--------------------+
only showing top 20 rows



In [8]:
user_artist_df = spark.read.option("delimiter", " ").csv(s3_path+'user_artist_data_small.txt', inferSchema=True)
column_names = ["user", "artist", "count"]  # Replace with your actual column names

# Add column names to the DataFrame
user_artist_df = user_artist_df.toDF(*column_names)

In [9]:
user_artist_df.show(10, False)

+-------+-------+-----+
|user   |artist |count|
+-------+-------+-----+
|1059637|1000010|238  |
|1059637|1000049|1    |
|1059637|1000056|1    |
|1059637|1000062|11   |
|1059637|1000094|1    |
|1059637|1000112|423  |
|1059637|1000113|5    |
|1059637|1000114|2    |
|1059637|1000123|2    |
|1059637|1000130|19129|
+-------+-------+-----+
only showing top 10 rows



### 2.2 Building a First Model

In [10]:
from pyspark.sql.functions import broadcast, when

all_data = (user_artist_df.join(broadcast(artist_by_id), 'artist', how='left') 
            # .withColumn('artist', when(col('alias').isNull(),col('artist')).otherwise(col('alias')))
            # .withColumn('artist',col('artist').cast(IntegerType())).drop('alias')
           )
all_data.count()

49481

In [11]:
all_data.show()
all_data.printSchema()

+-------+-------+-----+--------------------+
| artist|   user|count|                name|
+-------+-------+-----+--------------------+
|1000010|1059637|  238|           Aerosmith|
|1000049|1059637|    1|     Edna's Goldfish|
|1000056|1059637|    1|The Mighty Mighty...|
|1000062|1059637|   11|        Foo Fighters|
|1000094|1059637|    1|  The Bouncing Souls|
|1000112|1059637|  423|       Alkaline Trio|
|1000113|1059637|    5|         The Beatles|
|1000114|1059637|    2|           Pennywise|
|1000123|1059637|    2|             Incubus|
|1000130|1059637|19129|         Bright Eyes|
|1000139|1059637|    4|                Muse|
|1000241|1059637|  188|          Jason Mraz|
|1000263|1059637|  180|     Jimmy Eat World|
|1000289|1059637|    2|           Meat Loaf|
|1000305|1059637|    1| The Lightning Seeds|
|1000320|1059637|   21|                MxPx|
|1000340|1059637|    1|     At the Drive-In|
|1000427|1059637|   20|     New Found Glory|
|1000428|1059637|   12|         Blind Melon|
|1000433|1

In [52]:
train_data, test_data = all_data.randomSplit([0.9, 0.1],seed=42)

In [53]:
train_data.cache()
test_data.cache()

DataFrame[artist: int, user: int, count: int, name: string]

In [14]:
from pyspark.ml.recommendation import ALS
from pyspark.sql import Row

als_model = (ALS(rank=10, 
                 seed=0, 
                 maxIter=5, 
                 regParam=0.1, 
                 implicitPrefs=True, 
                 alpha=1.0, 
                 userCol='user', itemCol='artist', ratingCol='count', 
                 nonnegative=True, coldStartStrategy="drop")
            )

In [55]:
als_fitted = als_model.fit(train_data)
predictions = als_fitted.transform(test_data)
predictions.count()

2427

In [56]:
predictions.show(5, False)

+------+-------+-----+-------------+-----------+
|artist|user   |count|name         |prediction |
+------+-------+-----+-------------+-----------+
|100   |2007381|79   |Phoenix      |0.7957023  |
|202   |2007381|88   |Orbital      |0.7905948  |
|1183  |2007381|141  |Dido         |0.3429359  |
|1230  |2007381|65   |Joni Mitchell|0.048070427|
|1399  |2007381|358  |Madonna      |0.8430728  |
+------+-------+-----+-------------+-----------+
only showing top 5 rows



In [62]:
from pyspark.sql.functions import size, col, expr
true_labels = (predictions
                      .orderBy(col("user"), expr("count DESC"))
                      .groupBy("user")
                      .agg(expr("collect_list(artist) as artists"))
                      .withColumn('len', size(col('artists')))
                     )

In [31]:
perUserPredictions.toPandas().head(5)

,user,artists,len
0,2007381,"[1004055, 1009773, 1067278, 1230, 1021539, 127...",33
1,1059637,"[1043801, 1260489, 1078665, 1191501, 1261496, ...",38
2,2288164,"[1254198, 1791, 1003686, 1014893, 2000819, 100...",28
3,1001440,"[1009870, 1017284, 1026659, 1199755, 1241755, ...",105
4,1024631,"[1080469, 1145871, 1117224, 2013227, 1003085, ...",256


In ALS (Alternating Least Squares) with implicit feedback in PySpark, the prediction values are not constrained to a specific range such as 0 to 1. The range of prediction values can vary depending on several factors including the model parameters, the scale of the input data, and the characteristics of the training data.

In [ ]:
# predicted_labels = predictions.filter(col('prediction') > 1 ).orderBy("userId", desc("prediction")).groupBy("userId").agg(collect_list("movieID").alias("predicted_items"))

In [ ]:
predicted_labels =  (predictions
                      .orderBy(col("user"), expr("prediction DESC"))
                      .groupBy("user")
                      .agg(expr("collect_list(name) as preartists"))
                      .withColumn('len', size(col('preartists')))
                     )
predicted_labels.show(5, False)

In [69]:
eval_df = (true_labels.join(predicted_labels, "user")
           .withColumn("true_items", col("artists").cast("array<double>"))
           .withColumn("predicted_items", col("preartists").cast("array<double>"))
          ) # Cast true_items and predicted_items to array<double>

In [70]:
from pyspark.ml.evaluation import RankingEvaluator

In [71]:
evaluator = RankingEvaluator(
    predictionCol="predicted_items",
    labelCol="true_items",
    metricName="meanAveragePrecisionAtK",
    k= 3  # You can specify the value of K (e.g., top 10 recommendations)
)

In [72]:
ndcg_at_k = evaluator.evaluate(eval_df)
print(f"NDCG at K: {ndcg_at_k}")

NDCG at K: 0.003401360544217688


In [73]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize Spark session
spark = SparkSession.builder.appName("EvaluationExample").getOrCreate()

# Create True Interactions DataFrame with count-based labels
true_interactions = spark.createDataFrame([
    (1, 10, 5),
    (1, 20, 0),
    (2, 20, 3),
    (2, 30, 0),
    (3, 10, 1),
    (3, 30, 2)
], ["userId", "itemId", "label"])

# Create Predictions DataFrame (normalized scores)
predict = spark.createDataFrame([
    (1, 10, 0.8),
    (1, 20, 0.2),
    (2, 20, 0.6),
    (2, 30, 0.1),
    (3, 10, 0.7),
    (3, 30, 0.5)
], ["userId", "itemId", "prediction"])


In [75]:
max_interaction_count = true_interactions.select(F.max("label")).collect()[0][0]
max_interaction_count



5

In [78]:
# Scale predictions to the range of true interaction counts
scaled_predictions = predict.withColumn("scaled_prediction", F.col("prediction") * max_interaction_count)

In [79]:
scaled_predictions.show(5)

+------+------+----------+-----------------+
|userId|itemId|prediction|scaled_prediction|
+------+------+----------+-----------------+
|     1|    10|       0.8|              4.0|
|     1|    20|       0.2|              1.0|
|     2|    20|       0.6|              3.0|
|     2|    30|       0.1|              0.5|
|     3|    10|       0.7|              3.5|
+------+------+----------+-----------------+
only showing top 5 rows



In [80]:
predictions_sorted = scaled_predictions.orderBy("userId", F.desc("scaled_prediction"))

predictions_sorted.show()

+------+------+----------+-----------------+
|userId|itemId|prediction|scaled_prediction|
+------+------+----------+-----------------+
|     1|    10|       0.8|              4.0|
|     1|    20|       0.2|              1.0|
|     2|    20|       0.6|              3.0|
|     2|    30|       0.1|              0.5|
|     3|    10|       0.7|              3.5|
|     3|    30|       0.5|              2.5|
+------+------+----------+-----------------+



In [ ]:
# Group predictions by user and collect predicted items
predictions_grouped = predictions_sorted.groupBy("userId") \
    .agg(F.collect_list(F.struct("itemId", "scaled_prediction")).alias("predictedItems"))

# Group true interactions by user and collect true items
true_items_grouped = true_interactions.groupBy("userId") \
    .agg(F.collect_list(F.struct("itemId", "label")).alias("trueItems"))

# Join true items with predicted items
eval_df = true_items_grouped.join(predictions_grouped, on="userId", how="inner")

In [82]:
predictions_grouped.show(3, False)

+------+----------------------+
|userId|predictedItems        |
+------+----------------------+
|1     |[{10, 4.5}, {20, 1.2}]|
|3     |[{30, 2.4}, {10, 2.1}]|
|2     |[{20, 2.8}, {30, 0.6}]|
+------+----------------------+



In [83]:
true_items_grouped = true_interactions.groupBy("userId") \
    .agg(F.collect_list(F.struct("itemId", "label")).alias("trueItems"))

true_items_grouped.show(3, False)

+------+------------------+
|userId|trueItems         |
+------+------------------+
|1     |[{10, 5}, {20, 0}]|
|2     |[{20, 3}, {30, 0}]|
|3     |[{10, 1}, {30, 2}]|
+------+------------------+



In [84]:
# Join true items with predicted items
eval_df = true_items_grouped.join(predictions_grouped, on="userId", how="inner")

In [86]:
eval_df.show(5, False)

+------+------------------+----------------------+
|userId|trueItems         |predictedItems        |
+------+------------------+----------------------+
|1     |[{10, 5}, {20, 0}]|[{10, 4.5}, {20, 1.2}]|
|3     |[{10, 1}, {30, 2}]|[{30, 2.4}, {10, 2.1}]|
|2     |[{20, 3}, {30, 0}]|[{20, 2.8}, {30, 0.6}]|
+------+------------------+----------------------+



In [91]:
from pyspark.ml.evaluation import RankingEvaluator

# Initialize RankingEvaluator
evaluator = RankingEvaluator(metricName="ndcgAtK", k=10, predictionCol= "predictedItems")



In [92]:
# Evaluate
ndcg = evaluator.evaluate(eval_df)
print(f"NDCG at K: {ndcg}")



IllegalArgumentException: requirement failed: Column predictedItems must be of type equal to one of the following types: [array<double>, array<double>] but was actually of type array<struct<itemId:bigint,prediction:double>>.

In [ ]:
# Optionally, evaluate Precision and Recall
precision_at_k = evaluator.evaluate(eval_df, {evaluator.metricName: "precisionAtK"})
recall_at_k = evaluator.evaluate(eval_df, {evaluator.metricName: "recallAtK"})

print(f"Precision at K: {precision_at_k}")
print(f"Recall at K: {recall_at_k}")